# Sampling a transit's posterior

A reviewed notebook for the Gravitas compute page. It reads one `gravitas.observation/1` time series, handed to it as the text `gravitas_input`, and its four settings as the text `gravitas_params`. The example is the TESS light curve of HD 209458. It asks a different question of the same data than a fit does: not "which value fits best, and how wide is the valley of chi-square around it", but "how probable is each value, given the data and a prior", answered by drawing samples.

The sampler is written here in NumPy, nothing else: the affine-invariant ensemble sampler of Goodman and Weare (2010), the stretch move with `a = 2`, in the parallel half-step form of Foreman-Mackey et al. (2013), on four independent ensembles of eight walkers. It is the sampler the Prompt 103 gate measured, and the notebook's own tests hold it to the cases that gate used: a correlated Gaussian, the twisted-Gaussian banana and a conjugate normal problem, each with an exact answer (`SAMPLER_GATE.md`).

The target is the transit of the Observatory's fit panel (the model is written again here in NumPy and agrees with the application's to the last digit of a double, with 16 quadrature nodes where the panel uses 32) with only the planet's size `k = Rp/R*` and the impact parameter `b` free. The epoch, the period, `a/R*` and the limb-darkening pair `q1 = 0.298`, `q2 = 0.277` are held fixed (the first three are settings; the values are what the panel's fit of all seven parameters found). The prior is uniform: `k` between 0.005 and 0.5, `b` between 0 and `1 + k`. The baseline flux is solved exactly at every step, as the panel does, so the target is the likelihood profiled over it. Because the light curve's scatter is about three times its error bars, the log-likelihood is `-chi-square / (2 s)` with `s` the reduced chi-square at the best point (the same scaling the panel's profile uses).

What it does, in order:

1. **A grid.** The two-dimensional posterior on a grid, found by repeated zooming, which gives the exact marginals to compare the sampler with, and the profile interval (the chi-square rise of 1 with the other parameter minimized) on the same grid.
2. **Sample.** Four ensembles, started over-dispersed around the grid's center, for the number of steps asked; the first half of each is discarded.
3. **Diagnose.** The split R-hat over the four ensembles' mean traces and over every walker, the integrated autocorrelation time (Sokal's automatic window), the effective sample size and the acceptance fraction. If any of them fails its limit (R-hat under 1.05, at least 400 effective samples, at least 50 autocorrelation times after the burn-in, an acceptance fraction between 0.1 and 0.9) the notebook **refuses to report an interval** and says which failed: raise the number of steps.
4. **Compare.** Each parameter's central 68 percent interval from the draws, from the grid and as a profile, and the ratio of the draws' width to the profile's.

What the comparison shows, and what it does not. With everything but `k` and `b` held fixed, the posterior's central 68 percent interval and the profile's chi-square-plus-one interval have almost the same width: the problem is close to Gaussian in these two parameters. They are not the same thing (one is a probability under a prior, the other a range of values that fit about as well) and they are much narrower than the Observatory's own intervals, because the fit panel refits `a/R*`, the limb darkening, the epoch and the period at every value of `k`. Holding them fixed does not make them known; it makes their uncertainty invisible.

In [ ]:
# Read the input and the four settings.
import json, hashlib
import numpy as np

obs = json.loads(gravitas_input)
params = json.loads(gravitas_params)
assert obs["format"] == "gravitas.observation" and obs["formatVersion"] == 1, "this notebook reads gravitas.observation/1"
assert obs["kind"] == "time-series", "this notebook reads a time series"
STEPS = int(round(float(params.get("steps", 8000))))
T0 = float(params.get("epoch", 2826.78164))
PERIOD = float(params.get("period", 3.52454))
ARS = float(params.get("aRs", 9.21))
assert 1000 <= STEPS <= 10000, "steps is between 1000 and 10000"
assert 0.5 <= PERIOD <= 30, "the period is between 0.5 and 30 days"
assert 3 <= ARS <= 30, "aRs is between 3 and 30"
Q1, Q2 = 0.298, 0.277
input_digest = hashlib.sha256(gravitas_input.encode("utf-8")).hexdigest()

def column(role, of=None):
    for c in obs["columns"]:
        if c["role"] == role and (of is None or c.get("of") == of):
            return c
    return None

xc = next(c for c in obs["columns"] if c["id"] == obs["time"]["column"])
assert xc["unit"] == "d", "the time column must be in days"
yc = column("value")
ec = column("uncertainty", of=yc["id"])
assert ec is not None, "the light curve needs an uncertainty column"
num = lambda c: np.array([np.nan if v is None else v for v in c["values"]], float)
t, f, e = num(xc), num(yc), num(ec)
masked = np.zeros(len(t), bool)
for m in obs["masks"]:
    masked[m["rows"]] = True
keep = np.isfinite(t) & np.isfinite(f) & np.isfinite(e) & (e > 0) & ~masked
t, f, e = t[keep], f[keep], e[keep]
level = np.median(f)
y, w = f / level, 1.0 / (e / level) ** 2
EXPOSURE = float(np.median(np.diff(t)))
N = len(t)
print(f"{N} points; epoch {T0}, period {PERIOD} d, a/R* {ARS} held fixed; {STEPS} steps")

In [ ]:
# The ensemble sampler (Goodman and Weare 2010; Foreman-Mackey et al. 2013), NumPy only, and its diagnostics.
import numpy as np

def run_ensembles(logp, start, steps, seed, a=2.0):
    """Stretch move on M independent ensembles in the parallel half-step form.
    start has shape (M, W, d); logp maps (n, d) to (n,) and returns -inf outside the prior.
    Returns the chain (steps, M, W, d) and each ensemble's acceptance fraction."""
    rng = np.random.default_rng(seed)
    X = np.array(start, float)
    M, W, d = X.shape
    h = W // 2
    lp = logp(X.reshape(-1, d)).reshape(M, W)
    if not np.all(np.isfinite(lp)):
        raise ValueError("a walker starts outside the prior")
    chain = np.empty((steps, M, W, d))
    accepted = np.zeros(M)
    for s in range(steps):
        for half in (0, 1):
            mine = slice(0, h) if half == 0 else slice(h, W)
            other = slice(h, W) if half == 0 else slice(0, h)
            n = mine.stop - mine.start
            partner = np.take_along_axis(X[:, other, :], rng.integers(0, W - n, size=(M, n))[:, :, None], axis=1)
            z = ((a - 1) * rng.random((M, n)) + 1) ** 2 / a
            Y = partner + z[:, :, None] * (X[:, mine, :] - partner)
            lpY = logp(Y.reshape(-1, d)).reshape(M, n)
            ok = np.log(rng.random((M, n))) < (d - 1) * np.log(z) + lpY - lp[:, mine]
            X[:, mine, :] = np.where(ok[:, :, None], Y, X[:, mine, :])
            lp[:, mine] = np.where(ok, lpY, lp[:, mine])
            accepted += ok.sum(axis=1)
        chain[s] = X
    return chain, accepted / (steps * W)

def split_rhat(seqs):
    """Split Gelman-Rubin statistic of an array (sequences, steps): each sequence is cut in half."""
    n = seqs.shape[1] // 2
    s = np.concatenate([seqs[:, :n], seqs[:, n:2 * n]], axis=0)
    within = s.var(axis=1, ddof=1).mean()
    between = n * s.mean(axis=1).var(ddof=1)
    return float(np.sqrt(((n - 1) / n * within + between / n) / within))

def autocorr_time(x, c=5.0):
    """Integrated autocorrelation time of one series, Sokal's automatic window."""
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    spectrum = np.fft.rfft(x, 2 * n)
    acf = np.fft.irfft(spectrum * np.conjugate(spectrum))[:n]
    acf = acf / acf[0]
    taus = 2 * np.cumsum(acf) - 1
    inside = np.arange(n) < c * taus
    window = int(np.argmin(inside)) if not inside.all() else n - 1
    return float(taus[window])

def diagnose(chain, burn):
    """Per parameter: R-hat over the ensembles' mean traces and over every walker, tau, effective
    sample size, steps per tau; and the limits the notebook holds a run to."""
    post = chain[burn:]
    steps, M, W, d = post.shape
    out = []
    for p in range(d):
        means = post[:, :, :, p].mean(axis=2)
        tau = float(np.mean([autocorr_time(means[:, m]) for m in range(M)]))
        out.append({
            "rhat_ensembles": split_rhat(means.T),
            "rhat_walkers": split_rhat(post[:, :, :, p].reshape(steps, -1).T),
            "tau": tau, "ess": steps * M * W / tau, "steps_per_tau": steps / tau,
        })
    return out

def refusals(diag, acceptance):
    """What fails the limits, as sentences; empty when the run may report intervals."""
    why = []
    for i, dg in enumerate(diag):
        if max(dg["rhat_ensembles"], dg["rhat_walkers"]) >= 1.05:
            why.append(f"parameter {i}: R-hat {max(dg['rhat_ensembles'], dg['rhat_walkers']):.3f} is not under 1.05")
        if dg["ess"] < 400:
            why.append(f"parameter {i}: {dg['ess']:.0f} effective samples are fewer than 400")
        if dg["steps_per_tau"] < 50:
            why.append(f"parameter {i}: {dg['steps_per_tau']:.0f} autocorrelation times after the burn-in are fewer than 50")
    if not 0.1 <= float(np.mean(acceptance)) <= 0.9:
        why.append(f"the acceptance fraction {float(np.mean(acceptance)):.2f} is outside 0.1 to 0.9")
    return why

In [ ]:
# The transit of the Observatory's fit panel (js/inference/transit.js) in NumPy, with k and b free, evaluated for many walkers at once.
cyc = (t - T0) / PERIOD
dt = (cyc - np.floor(cyc + 0.5)) * PERIOD
reach = PERIOD / (2 * np.pi) * np.arcsin(min(1.0, 1.5 / ARS)) + EXPOSURE / 2 + 0.01
near = np.abs(dt) <= reach
Wo, Syo, Y2 = w[~near].sum(), (w[~near] * y[~near]).sum(), (w * y * y).sum()
tn, yn, wn = t[near], y[near], w[near]
SUPER, NODES = 5, 16
phi = 2 * np.pi * (tn[:, None] + EXPOSURE * ((np.arange(SUPER) + 0.5) / SUPER - 0.5)[None, :] - T0) / PERIOD
SEP_A, SEP_C, FRONT = ARS**2 * np.sin(phi) ** 2, np.cos(phi) ** 2, np.cos(phi) > 0
U1, U2 = 2 * np.sqrt(Q1) * Q2, np.sqrt(Q1) * (1 - 2 * Q2)
TOTAL = np.pi * (1 - U1 / 3 - U2 / 6)
Fm = lambda mu: mu**2 / 2 - U1 * (mu**2 / 2 - mu**3 / 3) - U2 * (mu**2 / 2 - 2 * mu**3 / 3 + mu**4 / 4)
S = (np.arange(NODES) + 0.5) / NODES
CS, SN = (1 - np.cos(np.pi * S)) / 2, np.pi * np.sin(np.pi * S) / 2 / NODES

def chi2(k, b):
    """Chi-square of the transit for arrays of k and b (the baseline solved exactly)."""
    k, b = np.asarray(k, float), np.asarray(b, float)
    z = np.sqrt(SEP_A[None] + b[:, None, None] ** 2 * SEP_C[None])
    kk = np.broadcast_to(k[:, None, None], z.shape)
    hit = (z < 1 + kk) & FRONT[None]
    zz, k_ = z[hit], kk[hit]
    inner = np.where(zz < k_, 2 * np.pi * (Fm(1.0) - Fm(np.sqrt(np.maximum(0.0, 1 - np.minimum(1, k_ - zz) ** 2)))), 0.0)
    a = np.abs(zz - k_)
    wid = np.minimum(1, zz + k_) - a
    r = a[:, None] + wid[:, None] * CS
    dr = wid[:, None] * SN
    arc = 2 * r * np.arccos(np.clip((r * r + (zz * zz - k_ * k_)[:, None]) / (2 * r * np.where(zz > 0, zz, 1.0)[:, None]), -1, 1))
    m = 1 - np.sqrt(np.maximum(0.0, 1 - r * r))
    partial = np.where((wid > 0) & (zz > 0), ((1 - U1 * m - U2 * m * m) * arc * dr).sum(axis=1), 0.0)
    occ = np.zeros(z.shape)
    occ[hit] = np.minimum(1, (inner + partial) / TOTAL)
    shape = 1 - occ.mean(axis=2)
    num_, den = Syo + (wn * yn * shape).sum(axis=1), Wo + (wn * shape * shape).sum(axis=1)
    return Y2 - num_ * num_ / den

In [ ]:
# A grid of the posterior, found by zooming in on its support.
def grid_of(kl, kh, bl, bh, n):
    kg, bg = np.linspace(kl, kh, n), np.linspace(bl, bh, n)
    K, B = np.meshgrid(kg, bg, indexing="ij")
    X2 = np.full(K.shape, np.inf)
    ok = B < 1 + K
    X2[ok] = chi2(K[ok], B[ok])
    return kg, bg, X2

kl, kh, bl, bh = 0.04, 0.30, 0.0, 1.0
for _ in range(8):
    kg, bg, X2 = grid_of(kl, kh, bl, bh, 41)
    reduced = X2.min() / (N - 8)
    rise = (X2 - X2.min()) / reduced
    inside = rise <= 60
    ii, jj = np.where(inside.any(axis=1))[0], np.where(inside.any(axis=0))[0]
    dk, db = kg[1] - kg[0], bg[1] - bg[0]
    kl, kh = max(0.005, kg[ii[0]] - dk), min(0.5, kg[ii[-1]] + dk)
    bl, bh = max(0.0, bg[jj[0]] - db), min(1.5, bg[jj[-1]] + db)
    edge = min(rise[0].min(), rise[-1].min(), rise[:, 0].min(), rise[:, -1].min())
    if inside.sum() >= 400 and edge > 25:
        break
kg, bg, X2 = grid_of(kl, kh, bl, bh, 61)
reduced = float(X2.min() / (N - 8))
rise = (X2 - X2.min()) / reduced
mass = np.exp(-0.5 * rise)
mass /= mass.sum()
edge_mass = float(mass[0].sum() + mass[-1].sum() + mass[:, 0].sum() + mass[:, -1].sum())
marginal = [mass.sum(axis=1), mass.sum(axis=0)]
axes = [kg, bg]

def quantile(values, pm, qs):
    c = np.cumsum(pm) - pm / 2
    return np.interp(qs, c, values)

def crossing(values, p):
    """Where a profile of chi-square rise crosses 1 on each side of its minimum, interpolated in sqrt(rise)."""
    i0 = int(np.argmin(p))
    s = np.sqrt(np.maximum(p - p[i0], 0))
    ends = []
    for step in (-1, 1):
        i = i0
        while 0 <= i + step < len(s) and s[i + step] <= 1:
            i += step
        j = i + step
        if not 0 <= j < len(s):
            ends.append(float(values[i]))
        else:
            ends.append(float(values[i] + (1 - s[i]) / (s[j] - s[i]) * (values[j] - values[i])))
    return ends[0], ends[1], float(values[i0])

profiles = [crossing(kg, rise.min(axis=1)), crossing(bg, rise.min(axis=0))]
grid_pct = [quantile(a, m, [0.1587, 0.5, 0.8413]) for a, m in zip(axes, marginal)]
grid_mean = np.array([(a * m).sum() for a, m in zip(axes, marginal)])
grid_sd = np.sqrt([((a - mu) ** 2 * m).sum() for a, m, mu in zip(axes, marginal, grid_mean)])
print(f"grid: chi-square {X2.min():.2f}, reduced {reduced:.3f}, edge mass {edge_mass:.1e}")
print(f"grid k {grid_pct[0][1]:.5f} (+{grid_pct[0][2] - grid_pct[0][1]:.5f} -{grid_pct[0][1] - grid_pct[0][0]:.5f}), profile {profiles[0][0]:.5f} to {profiles[0][1]:.5f}")
print(f"grid b {grid_pct[1][1]:.4f} (+{grid_pct[1][2] - grid_pct[1][1]:.4f} -{grid_pct[1][1] - grid_pct[1][0]:.4f}), profile {profiles[1][0]:.4f} to {profiles[1][1]:.4f}")

In [ ]:
# Sample: four ensembles of eight walkers, started over-dispersed around the grid's center.
def logp(theta):
    k, b = theta[:, 0], theta[:, 1]
    ok = (k >= 0.005) & (k <= 0.5) & (b >= 0) & (b < 1 + k)
    out = np.full(len(k), -np.inf)
    out[ok] = -0.5 * chi2(k[ok], b[ok]) / reduced
    return out

M, W = 4, 8
rng = np.random.default_rng(20261012)
centers = grid_mean + 2 * grid_sd * rng.standard_normal((M, 1, 2))
start = centers + 3 * grid_sd * rng.standard_normal((M, W, 2))
chain, acceptance = run_ensembles(logp, start, STEPS, 20261013)
burn = STEPS // 2
diag = diagnose(chain, burn)
for name, dg in zip(("k", "b"), diag):
    print(f"{name}: R-hat {dg['rhat_ensembles']:.3f} (ensembles), {dg['rhat_walkers']:.3f} (walkers); tau {dg['tau']:.1f}; {dg['ess']:.0f} effective samples; {dg['steps_per_tau']:.0f} tau after the burn-in")
print(f"acceptance fraction {float(np.mean(acceptance)):.2f}")
why = refusals(diag, acceptance)
if why:
    raise ValueError("the run is refused, so no interval is reported: " + "; ".join(why) + ". Raise the number of steps.")

In [ ]:
# Compare the draws with the grid and the profile, and return the numbers and the traces.
draws = chain[burn:].reshape(-1, 2)
pct = [np.percentile(draws[:, p], [15.87, 50, 84.13]) for p in range(2)]
width = [float(q[2] - q[0]) for q in pct]
ratio = [width[p] / (profiles[p][1] - profiles[p][0]) for p in range(2)]
offset = float(max(abs(pct[p][1] - grid_pct[p][1]) / grid_sd[p] for p in range(2)))
LICENSE = obs["license"]
ci = lambda q: {"kind": "interval", "lo": float(q[0]), "hi": float(q[2]), "level": 0.6827, "basis": "scaled"}
prof_ci = lambda p: {"kind": "interval", "lo": float(p[0]), "hi": float(p[1]), "level": 0.6827, "basis": "profile"}
none = {"kind": "none"}
quantities = [
    {"id": "k", "value": float(pct[0][1]), "unit": "", "origin": "fitted", "uncertainty": ci(pct[0])},
    {"id": "b", "value": float(pct[1][1]), "unit": "", "origin": "fitted", "uncertainty": ci(pct[1])},
    {"id": "k.profile", "value": profiles[0][2], "unit": "", "origin": "fitted", "uncertainty": prof_ci(profiles[0])},
    {"id": "b.profile", "value": profiles[1][2], "unit": "", "origin": "fitted", "uncertainty": prof_ci(profiles[1])},
    {"id": "k.width.ratio", "value": float(ratio[0]), "unit": "", "origin": "derived", "uncertainty": none},
    {"id": "b.width.ratio", "value": float(ratio[1]), "unit": "", "origin": "derived", "uncertainty": none},
    {"id": "grid.offset", "value": offset, "unit": "", "origin": "derived", "uncertainty": none},
    {"id": "rhat.max", "value": float(max(max(d_["rhat_ensembles"], d_["rhat_walkers"]) for d_ in diag)), "unit": "", "origin": "derived", "uncertainty": none},
    {"id": "ess.min", "value": float(min(d_["ess"] for d_ in diag)), "unit": "count", "origin": "derived", "uncertainty": none},
    {"id": "tau.max", "value": float(max(d_["tau"] for d_ in diag)), "unit": "count", "origin": "derived", "uncertainty": none},
    {"id": "acceptance", "value": float(np.mean(acceptance)), "unit": "", "origin": "derived", "uncertainty": none},
    {"id": "chi2.reduced", "value": reduced, "unit": "", "origin": "derived", "uncertainty": none},
    {"id": "steps", "value": float(STEPS), "unit": "count", "origin": "assumed", "uncertainty": none},
]
envelope = {
    "format": "gravitas.artifact", "formatVersion": 1,
    "id": f"compute:posterior-sampling:{input_digest[:16]}",
    "made": {"app": "gravitas"},
    "source": {"kind": "analysis", "id": "compute/posterior-sampling", "version": "1", "digest": input_digest},
    "provenance": {
        "credit": obs["credit"], "license": LICENSE, "retrieved": obs["retrieved"],
        "citations": [c["text"] for c in obs["citations"]],
        "reductions": obs["reductions"] + [
            "Normalized by the median. Transit model of the Observatory's fit panel (circular orbit, quadratic limb darkening, 16 quadrature nodes, 5 sub-exposures of 20 minutes), baseline solved exactly; k and b free, epoch, period, a/R*, q1 and q2 fixed.",
            f"Uniform prior: k in 0.005 to 0.5, b in 0 to 1 + k. Log-likelihood is minus chi-square over 2 s, s = {reduced:.3f} the reduced chi-square at the best point.",
            f"Affine-invariant ensemble sampler (stretch move, a = 2), 4 ensembles of 8 walkers, {STEPS} steps, the first {burn} discarded; seeded.",
        ],
    },
    "quantities": quantities,
    "warnings": [
        "only k and b are free: the epoch, the period, a/R*, q1 and q2 are held fixed, so these intervals are conditional on them and are much narrower than a fit that frees them",
        "a posterior interval is a probability under the prior and the scaled likelihood; a profile interval is the range of values whose chi-square is within 1 of the best. Here they nearly agree because the problem is nearly Gaussian in k and b; that is a finding about this problem and not a rule",
        "the likelihood's scale is the reduced chi-square, because the light curve's scatter is larger than its error bars; correlated noise is not modeled",
    ],
}
thin = max(1, (STEPS - burn) // 200)
index = np.arange(burn, STEPS, thin)
trace = chain[index].mean(axis=2)  # (rows, ensembles, parameters)
table = {
    "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
    "id": f"{obs['id']}:traces", "title": f"{obs['title']}: each ensemble's mean k and b, step by step after the burn-in",
    "object": obs["object"], "facility": obs["facility"], "origin": "model",
    "source": {"kind": "import", "id": obs["id"], "version": obs["source"].get("version")},
    "credit": obs["credit"], "license": LICENSE, "retrieved": obs["retrieved"],
    "citations": obs["citations"],
    "reductions": envelope["provenance"]["reductions"],
    "axes": {"x": "step", "y": "k1"},
    "columns": [{"id": "step", "name": "step", "unit": "count", "role": "x", "values": [float(i) for i in index]}]
    + [{"id": f"k{m + 1}", "name": f"k, ensemble {m + 1}", "unit": "", "role": "value", "values": [float(v) for v in trace[:, m, 0]]} for m in range(M)]
    + [{"id": f"b{m + 1}", "name": f"b, ensemble {m + 1}", "unit": "", "role": "value", "values": [float(v) for v in trace[:, m, 1]]} for m in range(M)],
    "masks": [], "annotations": [],
}
gravitas_output = {"envelope": envelope, "table": table}
print(f"k {pct[0][1]:.5f}, b {pct[1][1]:.4f}; width ratios {ratio[0]:.2f}, {ratio[1]:.2f}; offset from the grid {offset:.3f} sd; {len(index)} trace rows")